In [ ]:
"""
AutoEIT Automated Scoring Script 
Evaluation Test Submission by Shreya Khanna for GSoC 2026
--------------------------------
This script reads EIT transcription data from Excel, applies meaning-based scoring
using the Ortega (2000) rubric via the Gemini API, and writes sentence-level scores
to a new Excel file. The script includes retry logic, logging, and summary statistics
to ensure reproducibility and robustness.

Reproducibility Notes:
- Model: Gemma 3 27B (I tried using multiple Gemini models but they hit their rate limit very soon (in the logs as well), despite trying different approaches with my code, like batching and adding delays, so I settled on the 27B model which allowed me to complete the scoring without hitting the limit)
- Temperature: 0 (deterministic scoring)
- Rate limit delay: 2 seconds
- Input: Excel file with stimulus and transcription
- Output: Excel file with sentence-level scores
- Errors handled with retry logic
"""

import os
import json
import time
import openpyxl
import google.generativeai as genai

# ── Configuration ──────────────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")

INPUT_FILE  = "AutoEIT_Sample_Transcriptions_for_Scoring.xlsx"
OUTPUT_FILE = "AutoEIT_Scored_Output.xlsx"

MODEL_NAME = "gemma-3-27b-it"
REQUEST_DELAY = 2
MAX_RETRIES = 3
LOG_FILE = "scoring_log.txt"

# ── Rubric Context ─────────────────────────────────────────────────────────────
RUBRIC = """
You are an expert scorer for the Elicited Imitation Task (EIT) in Spanish linguistics research.
Apply the following meaning-based scoring rubric (Ortega, 2000).

Score 0: Nothing, garbled speech, only 1 word repeated, only function words repeated,
or only 1-2 content words out of order plus extraneous words not in original.
Score 1: About half of idea units represented but important info missing.
Score 2: More than half of idea units preserved but meaning slightly altered or incomplete.
Score 3: Original meaning preserved despite grammatical errors.
Score 4: Exact repetition.

Respond ONLY with JSON:
{
  "score": <0-4>,
  "reasoning": "<one sentence explanation>"
}
"""

# ── Logging ────────────────────────────────────────────────────────────────────
def log_message(message: str):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(message + "\n")

# ── Create Gemini Model ─────────────────────────────────────────────────────────
def create_model():
    genai.configure(api_key=GEMINI_API_KEY)
    return genai.GenerativeModel(
        MODEL_NAME,
        generation_config={
            "temperature": 0,
            "top_p": 0.0,
            "top_k": 1,
            "max_output_tokens": 200,
        }
    )

# ── Scoring Function ────────────────────────────────────────────────────────────
def score_item(model, sentence_num, stimulus, transcription):
    """
    Score a single transcription item.
    """

    clean_stimulus = stimulus.split("(")[0].strip()
    prompt = f"""
Score the following learner transcription using the EIT rubric.
Return ONLY a JSON object in this format:

{{"score": <0-4>}}

Stimulus: "{clean_stimulus}"
Transcription: "{transcription}"
"""

    for attempt in range(MAX_RETRIES):
        try:
            response = model.generate_content(RUBRIC + "\n" + prompt)
            text = response.text.strip()

            if text.startswith("```"):
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:]

            result = json.loads(text.strip())
            return result["score"]

        except Exception as e:
            print(f"Attempt {attempt+1} failed for sentence {sentence_num}: {e}")
            time.sleep(10 * (attempt + 1))

    return -1  # Return -1 if all retries fail

# ── Main Processing ─────────────────────────────────────────────────────────────
def process_workbook():
    """
    Reads Excel file, scores each transcription individually,
    writes results to new Excel file, and prints summary statistics.
    """
    model = create_model()

    wb = openpyxl.load_workbook(INPUT_FILE)
    wb_out = openpyxl.load_workbook(INPUT_FILE)

    participant_sheets = [s for s in wb.sheetnames if s != "Info"]
    all_results = {}

    for sheet_name in participant_sheets:
        print(f"\nProcessing participant: {sheet_name}")
        ws_in  = wb[sheet_name]
        ws_out = wb_out[sheet_name]

        sheet_results = []

        for row_idx, row in enumerate(ws_in.iter_rows(min_row=2, values_only=True), start=2):
            if not row[0]:
                continue

            sentence_num = row[0]
            stimulus = row[1]
            transcription = row[2]

            if not stimulus or not transcription:
                continue

            score = score_item(model, sentence_num, stimulus, transcription)
            ws_out.cell(row=row_idx, column=4, value=score)

            sheet_results.append({
                "sentence": sentence_num,
                "score": score
            })

            print(f"Sentence {sentence_num} → Score {score}")
            time.sleep(REQUEST_DELAY)

        wb_out.save(OUTPUT_FILE)
        all_results[sheet_name] = sheet_results
        print(f"Saved progress for {sheet_name}")

    # Summary stats
    print("\nSUMMARY")
    for sheet_name, results in all_results.items():
        valid = [r["score"] for r in results if r["score"] >= 0]
        if valid:
            avg = sum(valid) / len(valid)
            dist = {i: valid.count(i) for i in range(5)}
            print(f"{sheet_name}: Mean={avg:.2f}, Distribution={dist}")

    print(f"\nScores written to {OUTPUT_FILE}")

# ── Run Script ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("AutoEIT Scoring Script")
    print("-" * 40)

    if GEMINI_API_KEY == "YOUR_API_KEY_HERE":
        print("Set your GEMINI_API_KEY environment variable.")
    else:
        process_workbook()


C:\Users\khann\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\khann\AppData\Local\Temp\ipykernel_21860\1538307199.py:14: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


AutoEIT Scoring Script
----------------------------------------

Processing participant: 38001-1A
Sentence 1 → Score 4
Sentence 2 → Score 4
Sentence 3 → Score 4
Sentence 4 → Score 4
Sentence 5 → Score 2
Sentence 6 → Score 3
Sentence 7 → Score 3
Sentence 8 → Score 3
Sentence 9 → Score 3
Sentence 10 → Score 4
Sentence 11 → Score 1
Sentence 12 → Score 4
Sentence 13 → Score 3
Sentence 14 → Score 2
Sentence 15 → Score 4
Sentence 16 → Score 2
Sentence 17 → Score 2
Sentence 18 → Score 3
Sentence 19 → Score 1
Sentence 20 → Score 3
Sentence 21 → Score 2
Sentence 22 → Score 3
Sentence 23 → Score 3
Sentence 24 → Score 3
Sentence 25 → Score 0
Sentence 26 → Score 3
Sentence 27 → Score 4
Sentence 28 → Score 3
Sentence 29 → Score 3
Sentence 30 → Score 3
Saved progress for 38001-1A

Processing participant: 38002-2A
Sentence 1 → Score 4
Sentence 2 → Score 4
Sentence 3 → Score 2
Sentence 4 → Score 4
Sentence 5 → Score 0
Sentence 6 → Score 1
Sentence 7 → Score 2
Sentence 8 → Score 2
Sentence 9 → Score 1
